In [ ]:
# Exercício de Análise de Dados

#Importar as bibliotecas necessárias
import pandas as pd
import re
from datetime import datetime  # para trabalhar com datas
#pip install openpyxl# No terminal, para exportar para Excel com múltiplas abas, é necessário ter a biblioteca openpyxl instalada.

#Confirmar que as bibliotecas foram importadas corretamente
print('✓ pandas versão:', pd.__version__)
print('✓ Bibliotecas importadas com sucesso!')

#Abrir arqvuivo direto por URL
# URL = "https://raw.githubusercontent.com/Auladehoje/turma-visualizacao-de-dados/main/datasets/base_rh.csv"


#Abrir arquivo Local
df = r"C:\Users\Bruno\Documents\GitHub\Auladehoje\turma-visualizacao-de-dados\datasets\base_rh.csv"
df = pd.read_csv(df, sep=";", encoding="cp1252", decimal=",")


# -- Exploração dos dados --
## -- DIAGNÓSTICO -- 
### -- Confirmar os dados foram carregados corretamente -- 

print('Colunas do dataset:')
print(df.columns.tolist())

df['Salario'] = pd.to_numeric(df['Salario'], errors='coerce')
print(df['Salario'].dtype)
df.head()

df.info() # Exibe informações sobre o DataFrame, como tipo de dados e valores
df.describe() # Exibe estatísticas descritivas para colunas numéricas do DataFrame


##* Os três problemas que encontrei foram: 
# *1) As colunas possuem números Inteiros (int64), números decimais (float64), texto (string) e data (datetime). 
# *2) O Dataframe possui um total de 11 colunas, que precisam ser segregadas de acordo com o tipo de análise a ser feita, como ex: estatística descritiva, estatísticas variáveis, agrupamentos, etc.
# *3) A coluna de Data de Admissão não precisa entrar na estatística descritiva, pois não é uma variável numérica, mas sim uma variável temporal/informativa.


# Passo 3: converter datas e criar colunas derivadas
df['Data de Admissão'] = pd.to_datetime(df['Data_Admissao'], format='%d/%m/%Y', errors='coerce')

# Antes da conversão:
print(f'Tipo ANTES : {df["Data_Admissao"].dtype}')
print(f'Valor antes: "{df["Data_Admissao"].iloc[0]}"')
# .iloc[0] → acessa a primeira linha pelo índice (iloc = integer location)

print()


#* Comando de Cópiar para não alterar o df original
df_limpo = df.copy()


df_limpo['Data_Admissao'] = pd.to_datetime(
    df_limpo['Data_Admissao'],
    format='%d/%m/%Y',  # formato da string de entrada
    errors='coerce'     # datas inválidas viram NaT (Not a Time) em vez de erro
)

print(f'Tipo DEPOIS: {df_limpo["Data_Admissao"].dtype}')
print(f'Valor depois: {df_limpo["Data_Admissao"].iloc[0]}')
print(f'NaT gerados : {df_limpo["Data_Admissao"].isna().sum()}')


#* Criar colunas derivadas de data
df_limpo['Ano_Admissao'] = df_limpo['Data_Admissao'].dt.year
df_limpo['Mes_Admissao'] = df_limpo['Data_Admissao'].dt.month
df_limpo['Trimestre']    = df_limpo['Data_Admissao'].dt.quarter


df_limpo['Tempo_Casa_Anos'] = (
    (hoje_ts - df_limpo['Data_Admissao']).dt.days / 365.25
).round(1)
# Subtração de datas → timedelta
# .dt.days → extrai a diferença em dias
# / 365.25 → converte dias em anos
# .round(1) → arredonda para 1 casa decimal

#* Criar as Colunas - Ver o resultado
cols = ['Nome', 'Data_Admissao', 'Ano_Admissao', 'Mes_Admissao', 'Trimestre', 'Tempo_Casa_Anos']
df_limpo[cols].head(15)


#* Função *groupby* — maior e menor média de tempo de casa

def classificar_tempo(anos):
    """
    Recebe: anos (número) — o tempo de casa do funcionário
    Retorna: uma categoria em texto
    """
    # pd.isna() verifica se o valor é nulo (NaN ou NaT)
    if pd.isna(anos):
        return 'Sem data'
    elif anos < 1:
        return 'Menos de 1 ano'
    elif anos < 3:
        return '1 a 3 anos'
    elif anos < 5:
        return '3 a 5 anos'
    else:
        return 'Mais de 5 anos'


# Testar a função com valores individuais antes de aplicar no DataFrame
print('Testando a função:')
print(f'  0.5 anos → "{classificar_tempo(0.5)}"')
print(f'  2.1 anos → "{classificar_tempo(2.1)}"')
print(f'  4.8 anos → "{classificar_tempo(4.8)}"')
print(f'  9.0 anos → "{classificar_tempo(9.0)}"')


# Tempo médio de casa por departamento
media_depto = (
    df_limpo
    .groupby('Departamento')['Tempo_Casa_Anos']
    .mean()
    .round(1)
    .sort_values(ascending=False)
    .rename('Media_Anos')   # .rename() → renomeia a coluna resultante
    .reset_index()          # .reset_index() → converte o índice em coluna comum
)

print('Tempo médio de casa por departamento:')
print(media_depto.to_string(index=False))

# .idxmax() → retorna o NOME do índice com maior valor
maior = media_depto.set_index('Departamento')['Media_Anos'].idxmax()
menor = media_depto.set_index('Departamento')['Media_Anos'].idxmin()
print(f'\n→ Maior retenção: {maior}')
print(f'→ Menor retenção: {menor}')


#** O departamento com maior retenção é o de "RH" e o departamento com menor retenção é o de "Fianceiro".


# Tratamento do arquivo antes para salvar em .csv

df_limpo.to_csv(
    'base_rh_dia01.csv',
    index=False,         # não salva o índice numérico como coluna extra
    sep=';',             # separador ponto-e-vírgula
    encoding='utf-8-sig'
)

print('✓ base_rh_dia01.csv salvo!')
print(f'  Colunas finais: {df_limpo.columns.tolist()}')
print(f'  Linhas        : {len(df_limpo)}')


# Passo 5: exportar para Excel
# Exportação simples: tudo em uma aba
df.to_excel('base_rh.xlsx', index=False, sheet_name='Funcionarios')
print('✓ base_rh.xlsx (aba única) gerado!')


# Exportação avançada: uma aba por departamento
with pd.ExcelWriter('base_rh_por_depto.xlsx', engine='openpyxl') as writer:

    # Aba 1: dataset completo
    df.to_excel(writer, sheet_name='Todos', index=False)
    print('  Aba Todos: todos os registros')

    # df['Departamento'].unique() → lista de departamentos sem repetição
    # sorted() → ordena em ordem alfabética
    for depto in sorted(df['Departamento'].unique()):

        # Filtrar só os funcionários desse departamento
        df_depto = df[df['Departamento'] == depto]

        # Limite do Excel: nomes de aba têm no máximo 31 caracteres
        nome_aba = depto[:31]

        df_depto.to_excel(writer, sheet_name=nome_aba, index=False)
        print(f'  Aba "{nome_aba}": {len(df_depto)} registros')

print('\n✓ base_rh_por_depto.xlsx gerado!')

#Foi gerada a planilha 'base_rh_por_depto.xlsx' com uma aba para cada departamento, contendo os registros correspondentes a cada um.    



# Transformar o Arquivo em formato JSON
import json  # biblioteca nativa do Python para manipular JSON

# df.to_json() → converte o DataFrame para JSON e salva no arquivo
df.to_json(
    'base_rh.json',
    orient='records',      
    force_ascii=False,     # False = salva ã, é, ç como estão
    indent=2               # 2 espaços de indentação → arquivo legível
)

print('✓ base_rh.json gerado!')

# Ler de volta para confirmar
df_json = pd.read_json('base_rh.json')
print(f'  Linhas lidas do JSON: {len(df_json)}')

# Visualizar os 2 primeiros registros no formato JSON
with open('base_rh.json', 'r', encoding='utf-8') as f:
    amostra = json.load(f)
print('\nPrimeiros 2 registros no JSON:')
print(json.dumps(amostra[:2], ensure_ascii=False, indent=2))


# 1. Criar o objeto ExcelWriter
with pd.ExcelWriter('base_rh_deptos.xlsx', engine='openpyxl') as writer:
    
    # 2. Salvar a base inteira na aba 'Completo'
    df.to_excel(writer, sheet_name='Completo', index=False)
    
    # 3. Criar uma aba para cada departamento automaticamente
    # O comando df['Departamento'].unique() pega a lista de todos os setores sem repetir
    for depto in df['Departamento'].unique():
        # Filtra o dataframe apenas com as linhas do departamento atual
        df_temp = df[df['Departamento'] == depto]
        
        # Salva o resultado em uma nova aba com o nome do departamento
        # Usamos .to_excel dentro do loop para criar as abas dinamicamente
        df_temp.to_excel(writer, sheet_name=depto, index=False)

print("Arquivo 'base_rh_deptos.xlsx' gerado com sucesso!")

# Conferir a saída da nova planilha
df_deptos = pd.read_excel('base_rh_deptos.xlsx', sheet_name='RH')
print(df_deptos.head())



56b8090 (HEAD) Merge branch 'master' of https://github.com/cfneves/turma-visualizacao-de-dados into HEAD
ae5af3b (origin/master, origin/HEAD) Fiz progreção de execurção na base
33fe76b Minha mudança


###---FIM---###

#Aluno: Bruno Briani de Paula#
#Data:08/05/2026#


#REcarregar o arquivo ja processado #
# Grave o resultado processado em disco #



SyntaxError: invalid decimal literal (3985183312.py, line 234)